<a href="https://colab.research.google.com/github/ankitarchoudhary/IBM-Data-Science-Capstone/blob/main/Phase2_Data_Wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2: Data Wrangling - SpaceX Falcon 9 Capstone

This phase takes the raw API-derived dataset and:
1. Explores its structure and missing values
2. Converts the landing `Outcome` text into a clean binary label (`Class`): 1 = the first stage
   landed successfully, 0 = it did not
3. Saves a clean, wrangled dataset ready for exploratory data analysis (EDA)




## Load the Data

In [5]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

df = pd.read_csv('dataset_part_1.csv')
df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


In [6]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'drive', 'dataset_part_1.csv', 'sample_data']


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Explanation:**
- `pandas` and `numpy` are imported for tabular data handling and numerical operations.
- `pd.read_csv('dataset_part_1.csv')` loads the CSV file saved in Phase 1. This assumes the file
  has already been uploaded to Colab's Files panel (top-left folder icon).
- `df.head()` displays the first 5 rows to confirm the data loaded correctly.

In [8]:
df.shape

(90, 18)

**Explanation:** `.shape` returns (number of rows, number of columns).

In [9]:
df.dtypes

,0
FlightNumber,int64
Date,object
BoosterVersion,object
PayloadMass,float64
Orbit,object
LaunchSite,object
Outcome,object
Flights,int64
GridFins,bool
Reused,bool


**Explanation:** `.dtypes` lists the data type of every column. This is worth checking before
any calculations, since a column that looks numeric but is stored as text (`object`) would need
converting first.

## Check for Missing Values

In [10]:
df.isnull().sum()

,0
FlightNumber,0
Date,0
BoosterVersion,0
PayloadMass,0
Orbit,0
LaunchSite,0
Outcome,0
Flights,0
GridFins,0
Reused,0


**Explanation:** counts missing values per column. `LandingPad` is expected to have missing
values - it only has a value when a launch attempted a drone-ship or ground-pad landing; launches
with no landing attempt, or an ocean landing, do not have a landing pad at all, so `NaN` here is
meaningful, not an error to be "fixed".

In [11]:
percentage_missing = df.isnull().sum() / len(df) * 100
percentage_missing

,0
FlightNumber,0.000000
Date,0.000000
BoosterVersion,0.000000
PayloadMass,0.000000
Orbit,0.000000
LaunchSite,0.000000
Outcome,0.000000
Flights,0.000000
GridFins,0.000000
Reused,0.000000


**Explanation:** dividing the missing count by the total number of rows (`len(df)`) and
multiplying by 100 expresses missing values as a percentage, which is often easier to interpret
than a raw count, especially when comparing datasets of different sizes.

## Explore Categorical Columns

In [12]:
df['LaunchSite'].value_counts()

,count
LaunchSite,
CCAFS SLC 40,55
KSC LC 39A,22
VAFB SLC 4E,13


**Explanation:** `.value_counts()` counts how many launches occurred at each site, in
descending order.

In [13]:
df['Orbit'].value_counts()

,count
Orbit,
GTO,27
ISS,21
VLEO,14
PO,9
LEO,7
SSO,5
MEO,3
HEO,1
ES-L1,1


**Explanation:** same idea, applied to the `Orbit` column - this shows which orbit types
are most common in the dataset.

## Understand the Outcome Column

The `Outcome` column combines two pieces of information as text (for example, `"True ASDS"` or
`"False Ocean"`): whether the landing succeeded, and what type of landing was attempted (ASDS =
drone ship, RTLS = return to launch site, Ocean = controlled ocean landing, or no attempt at all).

In [14]:
df['Outcome'].value_counts()

,count
Outcome,
True ASDS,41
None None,19
True RTLS,14
False ASDS,6
True Ocean,5
False Ocean,2
None ASDS,2
False RTLS,1


**Explanation:** lists every distinct value in `Outcome` along with how many times it
appears, which is needed to decide which outcomes count as a successful landing and which do not.

In [15]:
bad_outcomes = {'False ASDS', 'False Ocean', 'None None', 'None ASDS', 'False RTLS'}
bad_outcomes

{'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}

**Explanation:** this defines the set of `Outcome` values that represent an **unsuccessful**
landing (or no landing attempt at all):
- `False ASDS` / `False RTLS` / `False Ocean` - a landing was attempted but failed
- `None None` - no core data was available at all
- `None ASDS` - a drone-ship landing was planned, but the outcome is unknown/unrecorded

Any `Outcome` value **not** in this set (i.e. `True ASDS`, `True RTLS`, `True Ocean`) represents a
successful landing.

In [16]:
df['Class'] = df['Outcome'].apply(lambda outcome: 0 if outcome in bad_outcomes else 1)
df[['Outcome', 'Class']].head(10)

,Outcome,Class
0,None None,0
1,None None,0
2,None None,0
3,False Ocean,0
4,None None,0
5,None None,0
6,True Ocean,1
7,True Ocean,1
8,None None,0
9,None None,0


**Explanation:**
- `.apply(lambda outcome: ...)` runs a small function on every value in the `Outcome` column.
- The lambda checks whether each `Outcome` value is in the `bad_outcomes` set: if it is, the
  result is `0` (unsuccessful landing); otherwise, it is `1` (successful landing).
- This creates the `Class` column, which is the target variable used later for prediction.
- If `dataset_part_1.csv` already included a `Class` column (for example, if it came from the
  IBM fallback dataset), this recomputes it directly from `Outcome` and overwrites it - this is a
  useful check to confirm the two agree.

In [17]:
df['Class'].value_counts()

,count
Class,
1,60
0,30


**Explanation:** shows how many launches are labeled successful (1) versus unsuccessful (0)
overall - useful for understanding class balance before building a prediction model later.

In [18]:
success_rate = df['Class'].mean() * 100
print(f"Overall landing success rate: {success_rate:.2f}%")

Overall landing success rate: 66.67%


**Explanation:** since `Class` only contains 0s and 1s, `.mean()` conveniently gives the
proportion of successful landings (the average of 0s and 1s equals the fraction of 1s). Multiplying
by 100 converts this to a percentage.

## Save the Wrangled Dataset

In [19]:
df.to_csv('spacex_wrangled.csv', index=False)
print("Saved: spacex_wrangled.csv")
print("Shape:", df.shape)

Saved: spacex_wrangled.csv
Shape: (90, 18)
